# Stage 04c: Feature Encoding + Model Preparation

**Purpose:** Prepare ALL model-ready data

**Inputs:** data/04b_train.parquet, data/04b_test.parquet

**Outputs:**
- data/04c_train_encoded.parquet (after exclusions)
- data/04c_test_encoded.parquet (after exclusions)
- data/04c_train_glm.parquet (raw GLM predictions)
- data/04c_test_glm.parquet (raw GLM predictions)
- config_generated/04c_monotonicity_constraints.yaml

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_liab/v1"


In [3]:
import pandas as pd
import yaml
import os, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment, get_machine_config
from feature_encoder import apply_master_encoding, save_encoders
from model_utils import load_monotonicity_constraints, load_exclusions, load_glm_init, apply_feature_filters

print("########################################")
print("# STAGE 04c: ENCODING + MODEL PREP")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 04c: ENCODING + MODEL PREP
########################################


In [4]:
config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
print(f'Output: {output_base}')
vehicle_type = cfg["experiment"]["vehicle_type"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]
join_key = cfg["data"]["join_key"]
print(f"Vehicle type: {vehicle_type}")
print(f"Target: {target}")
print(f"Exposure: {exposure}")

Output: output/car_liab/v1
Vehicle type: CAR
Target: pp_bi
Exposure: ee_bi_imps


In [5]:
# Load train/test splits
train_file = f'{output_base}/data/04b_train.parquet'
test_file = f'{output_base}/data/04b_test.parquet'

print(f'\n* Loading data...')
train = pd.read_parquet(train_file)
test = pd.read_parquet(test_file)

print(f'  Train: {train.shape}')
print(f'  Test: {test.shape}')


* Loading data...


  Train: (7481727, 122)
  Test: (7483698, 122)


In [6]:
# Apply master encoding
print(f'\n* Applying master feature encoding...')

train_encoded, test_encoded, encoders, encoding_summary = apply_master_encoding(
    train, test, config_path,
    min_frequency=0.01,  # 1% threshold
    min_count=50         # OR 50 observations
)


* Applying master feature encoding...
[OK] Loaded master encoding: 406 columns



[OK] Encoding complete:
  Train shape: (7481727, 199)
  Test shape: (7483698, 199)


In [7]:
# Display encoding summary
print(f'\n* Encoding Summary:')
print(f'\nEncoding types used:')
print(encoding_summary['encoding_type'].value_counts())

print(f'\nCategories with __OTHER__:')
print(encoding_summary[encoding_summary['has_other']== True][['original_column', 'n_categories']].head(10))

print(f'\nCategories with __MISSING__:')
print(encoding_summary[encoding_summary['has_missing'] == True][['original_column', 'n_categories']].head(10))


* Encoding Summary:

Encoding types used:
encoding_type
ordinal_0_5                30
one_hot                    30
binary                     20
numeric                     5
remap_2to4_then_ordinal     5
skip                        2
DROP                        2
Name: count, dtype: int64

Categories with __OTHER__:
                             original_column  n_categories
9                              NumMinAcc_raw             6
10                           NumMajinAcc_raw             6
11                            NumSpdViol_raw             7
12                            NumMinViol_raw            11
13                            NumMajViol_raw             7
25  vc_active_collision_avoidance_system_raw             4
26          vc_active_driving_assistance_raw             4
27          vc_active_parking_assistance_raw             4
28            vc_adaptive_cruise_control_raw             4
30  vc_audible_forward_collision_warning_raw             4

Categories with __MISSING__:


In [8]:
# Apply feature exclusions
print(f"\n* Filtering features...")
exclusions = load_exclusions(config_path, vehicle_type)
all_features = list(train_encoded.columns)
filtered_features = apply_feature_filters(
    all_features, 
    exclusions, 
    target=target, 
    exposure=exposure, 
    join_key=join_key
)
train_encoded = train_encoded[filtered_features]
test_encoded = test_encoded[filtered_features]
print(f"  {len(all_features)} -> {len(filtered_features)} features")


* Filtering features...
Loaded 3 exclusions for CAR
Auto-excluding: target=pp_bi, exposure=ee_bi_imps, join_key=vin_date
Filtered 2 features: ['ee_bi_imps', 'vc_vehicle_immobilizer_raw']
  199 -> 197 features


In [9]:
# Prepare monotonicity constraints
print(f"\n* Preparing monotonicity constraints...")
mono_dict = load_monotonicity_constraints(config_path, vehicle_type, filtered_features)
os.makedirs(f"{output_base}/config_generated", exist_ok=True)
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
with open(mono_file, "w") as f:
    yaml.dump(mono_dict, f)
print(f"  Saved: {mono_file}")


* Preparing monotonicity constraints...
Loaded 31 monotonicity constraints for CAR
  Saved: output/car_liab/v1/config_generated/04c_monotonicity_constraints.yaml


In [10]:
# Load GLM predictions
if cfg.get("model", {}).get("use_glm_init", False):
    print(f"\n* Loading GLM predictions...")
    # Get machine-specific aux data path
    pc_num = open("current.pc").read().strip()
    pc_id = f"PC{pc_num}"
    aux_data_path = cfg["machines"][pc_id]["paths"]["aux_data_path"]
    control_file = cfg["data"]["control_model_file"]
    target_col = cfg["model"].get("glm_prediction_column", f"pred_{target}")
    
    # Load raw GLM predictions (will be used for residual method and base_margin)
    from model_utils import load_glm_predictions
    glm_pred_train = load_glm_predictions(aux_data_path, control_file, train, join_key, target_col)
    glm_pred_test = load_glm_predictions(aux_data_path, control_file, test, join_key, target_col)
    
    # Save raw GLM predictions (reset index to match encoded data)
    pd.DataFrame({"glm_pred": glm_pred_train.values}).to_parquet(f"{output_base}/data/04c_train_glm.parquet", index=False)
    pd.DataFrame({"glm_pred": glm_pred_test.values}).to_parquet(f"{output_base}/data/04c_test_glm.parquet", index=False)
    print(f"  Saved GLM prediction files")
else:
    print(f"\n* GLM init disabled")


* Loading GLM predictions...


Loaded GLM predictions: mean=217.4753, std=79.0180


Loaded GLM predictions: mean=217.7354, std=78.9644


  Saved GLM prediction files


In [11]:
# Save encoded data
train_output = f'{output_base}/data/04c_train_encoded.parquet'
test_output = f'{output_base}/data/04c_test_encoded.parquet'

train_encoded.to_parquet(train_output, index=False)
test_encoded.to_parquet(test_output, index=False)

print(f'\n* Saved:')
print(f'  {train_output}')
print(f'  {test_output}')


* Saved:
  output/car_liab/v1/data/04c_train_encoded.parquet
  output/car_liab/v1/data/04c_test_encoded.parquet


In [12]:
# Save encoders for holdout
save_encoders(encoders, encoding_summary, output_base)

[OK] Saved encoders: output/car_liab/v1/models/04c_encoders.pkl
[OK] Saved encoding summary: output/car_liab/v1/results/04c_encoding_summary.csv
[OK] Saved encoding map: output/car_liab/v1/models/04c_encoding_map.json


In [13]:
print("\n########################################")
print("# STAGE 04c: COMPLETE")
print("########################################")


########################################
# STAGE 04c: COMPLETE
########################################
